# Adaptive RAG

In [540]:
SAVE_PATH = 'results_qwen_4B_ENTROPY_THRESHOLD_2.json'

In [541]:
import json

with open(SAVE_PATH, 'r', encoding='utf-8') as f:
    results = json.load(f)

In [542]:
import numpy as np

def compute_metrics(results_by_main):
    total_subqs = 0
    correct_subqs = 0
    success_subqs = 0
    steps_per_subq = []
    initial_has_correct_stats = {'total': 0, 'correct': 0}
    initial_no_correct_stats = {'total': 0, 'correct': 0}

    main_stats = {}

    for main_id, main_data in results_by_main.items():
        # Check that main_data is a dict and contains 'subquestions'
        if not isinstance(main_data, dict):
            print(f"Warning: for main_id {main_id} data is not a dict ({type(main_data)}), skipping.")
            continue

        subqs = main_data.get('subquestions', {})
        if not isinstance(subqs, dict):
            print(f"Warning: for main_id {main_id} subquestions is not a dict ({type(subqs)}), skipping.")
            continue

        num_subqs = len(subqs)
        num_correct_main = 0

        for subq, res in subqs.items():
            if not isinstance(res, dict):
                print(f"Warning: for subquestion {subq} data is not a dict ({type(res)}), skipping.")
                continue

            total_subqs += 1

            # Determine if the answer is correct
            is_correct = res.get('is_correct', False)
            if is_correct:
                correct_subqs += 1
                num_correct_main += 1

            # Successful completion
            if res.get('success', False):
                success_subqs += 1

            # Number of steps
            steps = len(res.get('logits_history', []))
            steps_per_subq.append(steps)

            # Statistics for initial retrieval
            initial_has_correct = res.get('initial_has_correct', False)
            if initial_has_correct:
                initial_has_correct_stats['total'] += 1
                if is_correct:
                    initial_has_correct_stats['correct'] += 1
            else:
                initial_no_correct_stats['total'] += 1
                if is_correct:
                    initial_no_correct_stats['correct'] += 1

        main_stats[main_id] = {
            'num_subqs': num_subqs,
            'num_correct': num_correct_main,
            'all_correct': num_correct_main == num_subqs,
            'main_correct': main_data.get('main_correct', False)  # can be added for comparison
        }

    total_mains = len(main_stats)
    mains_all_correct = sum(1 for v in main_stats.values() if v['all_correct'])

    metrics = {
        'total_subquestions': total_subqs,
        'subq_accuracy': correct_subqs / total_subqs if total_subqs else 0,
        'subq_success_rate': success_subqs / total_subqs if total_subqs else 0,
        'avg_steps_per_subq': np.mean(steps_per_subq) if steps_per_subq else 0,
        'std_steps_per_subq': np.std(steps_per_subq) if steps_per_subq else 0,
        'min_steps_per_subq': min(steps_per_subq) if steps_per_subq else 0,
        'max_steps_per_subq': max(steps_per_subq) if steps_per_subq else 0,
        'total_main_questions': total_mains,
        'main_accuracy': mains_all_correct / total_mains if total_mains else 0,
        'main_all_correct_count': mains_all_correct,
        'subq_accuracy_when_initial_has_correct': (
            initial_has_correct_stats['correct'] / initial_has_correct_stats['total']
            if initial_has_correct_stats['total'] else 0
        ),
        'subq_accuracy_when_initial_no_correct': (
            initial_no_correct_stats['correct'] / initial_no_correct_stats['total']
            if initial_no_correct_stats['total'] else 0
        ),
        'main_stats': main_stats
    }

    return metrics

def print_metrics(metrics):
    """Pretty print of metrics."""
    print("\n" + "="*60)
    print("FINAL METRICS")
    print("="*60)
    print(f"Total subquestions: {metrics['total_subquestions']}")
    print(f"Subquestion accuracy: {metrics['subq_accuracy']:.2%}")
    print(f"Agent successful completion rate: {metrics['subq_success_rate']:.2%}")
    print(f"Average steps per subquestion: {metrics['avg_steps_per_subq']:.2f} ± {metrics['std_steps_per_subq']:.2f} "
          f"(min: {metrics['min_steps_per_subq']}, max: {metrics['max_steps_per_subq']})")
    print(f"\nTotal main questions: {metrics['total_main_questions']}")
    print(f"Main questions where all subquestions are answered correctly: {metrics['main_all_correct_count']} "
          f"({metrics['main_accuracy']:.2%})")
    print(f"\nSubquestion accuracy:")
    print(f"  - when the correct document was in the initial retrieval: {metrics['subq_accuracy_when_initial_has_correct']:.2%}")
    print(f"  - when the correct document was NOT in the initial retrieval: {metrics['subq_accuracy_when_initial_no_correct']:.2%}")

    print("\nBreakdown by main questions:")
    for main_id, stats in metrics['main_stats'].items():
        print(f"  Main {main_id}: {stats['num_correct']}/{stats['num_subqs']} correct subquestions "
              f"({'all correct' if stats['all_correct'] else 'not all'}) (main_correct={stats.get('main_correct', 'N/A')})")

In [543]:
metrics = compute_metrics(daresults_by_mainta)
print_metrics(metrics)


FINAL METRICS
Total subquestions: 246
Subquestion accuracy: 66.67%
Agent successful completion rate: 77.24%
Average steps per subquestion: 1.41 ± 1.11 (min: 0, max: 5)

Total main questions: 100
Main questions where all subquestions are answered correctly: 41 (41.00%)

Subquestion accuracy:
  - when the correct document was in the initial retrieval: 77.00%
  - when the correct document was NOT in the initial retrieval: 0.00%

Breakdown by main questions:
  Main 2hop__81825_49084: 2/2 correct subquestions (all correct) (main_correct=True)
  Main 2hop__131455_11960: 2/2 correct subquestions (all correct) (main_correct=True)
  Main 3hop1__694534_160088_85460: 1/3 correct subquestions (not all) (main_correct=False)
  Main 2hop__726717_610238: 1/2 correct subquestions (not all) (main_correct=False)
  Main 3hop1__106864_160713_77246: 2/3 correct subquestions (not all) (main_correct=False)
  Main 2hop__175168_110222: 2/2 correct subquestions (all correct) (main_correct=True)
  Main 2hop__135

In [544]:
SAVE_PATH = 'results_qwen_9B_ENTROPY_THRESHOLD_2.json'

In [545]:
import json

with open(SAVE_PATH, 'r', encoding='utf-8') as f:
    daresults_by_mainta = json.load(f)

In [546]:
metrics = compute_metrics(daresults_by_mainta)
print_metrics(metrics)


FINAL METRICS
Total subquestions: 246
Subquestion accuracy: 66.67%
Agent successful completion rate: 77.24%
Average steps per subquestion: 1.41 ± 1.11 (min: 0, max: 5)

Total main questions: 100
Main questions where all subquestions are answered correctly: 41 (41.00%)

Subquestion accuracy:
  - when the correct document was in the initial retrieval: 77.00%
  - when the correct document was NOT in the initial retrieval: 0.00%

Breakdown by main questions:
  Main 2hop__81825_49084: 2/2 correct subquestions (all correct) (main_correct=True)
  Main 2hop__131455_11960: 2/2 correct subquestions (all correct) (main_correct=True)
  Main 3hop1__694534_160088_85460: 1/3 correct subquestions (not all) (main_correct=False)
  Main 2hop__726717_610238: 1/2 correct subquestions (not all) (main_correct=False)
  Main 3hop1__106864_160713_77246: 2/3 correct subquestions (not all) (main_correct=False)
  Main 2hop__175168_110222: 2/2 correct subquestions (all correct) (main_correct=True)
  Main 2hop__135

# Naive RAG

In [547]:
import json

with open('results_qwen_9B_qwen.json', 'r', encoding='utf-8') as f:
    results_by_main = json.load(f)

In [548]:
# Подсчёт метрик
total_questions = len(results_by_main)
retriever_all_found = 0  # сколько раз ретривер нашёл все правильные параграфы
model_all_found = 0      # сколько раз модель (агент) нашла все правильные параграфы

total_retrieved_correct_sum = 0
total_found_correct_sum = 0
total_correct_count_sum = 0

for main_id, data in results_by_main.items():
    total_correct = data['total_correct_count']
    retrieved_correct = data['num_retrieved_correct']
    found_correct = data['num_found_correct']
    
    total_correct_count_sum += total_correct
    total_retrieved_correct_sum += retrieved_correct
    total_found_correct_sum += found_correct
    
    if retrieved_correct == total_correct:
        retriever_all_found += 1
    if found_correct == total_correct:
        model_all_found += 1

# Процент вопросов, где найдены ВСЕ правильные параграфы
retriever_perfect = retriever_all_found / total_questions * 100
model_perfect = model_all_found / total_questions * 100

# Среднее количество найденных параграфов (по всем вопросам)
avg_retrieved_correct = total_retrieved_correct_sum / total_questions
avg_found_correct = total_found_correct_sum / total_questions
avg_total_correct = total_correct_count_sum / total_questions

# Вывод
print("=== МЕТРИКИ ===")
print(f"Всего вопросов: {total_questions}")
print(f"Ретривер (top-30) нашёл все правильные параграфы в {retriever_all_found} вопросах ({retriever_perfect:.2f}%)")
print(f"Модель (LLM) нашла все правильные параграфы в {model_all_found} вопросах ({model_perfect:.2f}%)")
print(f"Среднее количество правильных параграфов на вопрос: {avg_total_correct:.2f}")
print(f"Среднее количество правильных параграфов, найденных ретривером: {avg_retrieved_correct:.2f}")
print(f"Среднее количество правильных параграфов, найденных моделью: {avg_found_correct:.2f}")

# Дополнительно: recall по параграфам (сколько процентов от всех правильных параграфов найдено)
recall_retriever = total_retrieved_correct_sum / total_correct_count_sum * 100
recall_model = total_found_correct_sum / total_correct_count_sum * 100
print(f"Recall ретривера: {recall_retriever:.2f}%")
print(f"Recall модели: {recall_model:.2f}%")

=== МЕТРИКИ ===
Всего вопросов: 100
Ретривер (top-30) нашёл все правильные параграфы в 65 вопросах (65.00%)
Модель (LLM) нашла все правильные параграфы в 38 вопросах (38.00%)
Среднее количество правильных параграфов на вопрос: 2.46
Среднее количество правильных параграфов, найденных ретривером: 2.00
Среднее количество правильных параграфов, найденных моделью: 1.34
Recall ретривера: 81.30%
Recall модели: 54.47%


In [549]:
import json

with open('results_qwen_9B_bm_25.json', 'r', encoding='utf-8') as f:
    results_by_main = json.load(f)

In [550]:
# Подсчёт метрик
total_questions = len(results_by_main)
retriever_all_found = 0  # сколько раз ретривер нашёл все правильные параграфы
model_all_found = 0      # сколько раз модель (агент) нашла все правильные параграфы

total_retrieved_correct_sum = 0
total_found_correct_sum = 0
total_correct_count_sum = 0

for main_id, data in results_by_main.items():
    total_correct = data['total_correct_count']
    retrieved_correct = data['num_retrieved_correct']
    found_correct = data['num_found_correct']
    
    total_correct_count_sum += total_correct
    total_retrieved_correct_sum += retrieved_correct
    total_found_correct_sum += found_correct
    
    if retrieved_correct == total_correct:
        retriever_all_found += 1
    if found_correct == total_correct:
        model_all_found += 1

# Процент вопросов, где найдены ВСЕ правильные параграфы
retriever_perfect = retriever_all_found / total_questions * 100
model_perfect = model_all_found / total_questions * 100

# Среднее количество найденных параграфов (по всем вопросам)
avg_retrieved_correct = total_retrieved_correct_sum / total_questions
avg_found_correct = total_found_correct_sum / total_questions
avg_total_correct = total_correct_count_sum / total_questions

# Вывод
print("=== МЕТРИКИ ===")
print(f"Всего вопросов: {total_questions}")
print(f"Ретривер (top-30) нашёл все правильные параграфы в {retriever_all_found} вопросах ({retriever_perfect:.2f}%)")
print(f"Модель (LLM) нашла все правильные параграфы в {model_all_found} вопросах ({model_perfect:.2f}%)")
print(f"Среднее количество правильных параграфов на вопрос: {avg_total_correct:.2f}")
print(f"Среднее количество правильных параграфов, найденных ретривером: {avg_retrieved_correct:.2f}")
print(f"Среднее количество правильных параграфов, найденных моделью: {avg_found_correct:.2f}")

# Дополнительно: recall по параграфам (сколько процентов от всех правильных параграфов найдено)
recall_retriever = total_retrieved_correct_sum / total_correct_count_sum * 100
recall_model = total_found_correct_sum / total_correct_count_sum * 100
print(f"Recall ретривера: {recall_retriever:.2f}%")
print(f"Recall модели: {recall_model:.2f}%")

=== МЕТРИКИ ===
Всего вопросов: 100
Ретривер (top-30) нашёл все правильные параграфы в 0 вопросах (0.00%)
Модель (LLM) нашла все правильные параграфы в 22 вопросах (22.00%)
Среднее количество правильных параграфов на вопрос: 2.46
Среднее количество правильных параграфов, найденных ретривером: 0.00
Среднее количество правильных параграфов, найденных моделью: 0.81
Recall ретривера: 0.00%
Recall модели: 32.93%


In [37]:
import json

with open('results_qwen_4B_bm_25.json', 'r', encoding='utf-8') as f:
    results_by_main = json.load(f)

In [38]:
# Подсчёт метрик
total_questions = len(results_by_main)
retriever_all_found = 0  # сколько раз ретривер нашёл все правильные параграфы
model_all_found = 0      # сколько раз модель (агент) нашла все правильные параграфы

total_retrieved_correct_sum = 0
total_found_correct_sum = 0
total_correct_count_sum = 0

for main_id, data in results_by_main.items():
    total_correct = data['total_correct_count']
    retrieved_correct = data['num_retrieved_correct']
    found_correct = data['num_found_correct']
    
    total_correct_count_sum += total_correct
    total_retrieved_correct_sum += retrieved_correct
    total_found_correct_sum += found_correct
    
    if retrieved_correct == total_correct:
        retriever_all_found += 1
    if found_correct == total_correct:
        model_all_found += 1

# Процент вопросов, где найдены ВСЕ правильные параграфы
retriever_perfect = retriever_all_found / total_questions * 100
model_perfect = model_all_found / total_questions * 100

# Среднее количество найденных параграфов (по всем вопросам)
avg_retrieved_correct = total_retrieved_correct_sum / total_questions
avg_found_correct = total_found_correct_sum / total_questions
avg_total_correct = total_correct_count_sum / total_questions

# Вывод
print("=== МЕТРИКИ ===")
print(f"Всего вопросов: {total_questions}")
print(f"Ретривер (top-30) нашёл все правильные параграфы в {retriever_all_found} вопросах ({retriever_perfect:.2f}%)")
print(f"Модель (LLM) нашла все правильные параграфы в {model_all_found} вопросах ({model_perfect:.2f}%)")
print(f"Среднее количество правильных параграфов на вопрос: {avg_total_correct:.2f}")
print(f"Среднее количество правильных параграфов, найденных ретривером: {avg_retrieved_correct:.2f}")
print(f"Среднее количество правильных параграфов, найденных моделью: {avg_found_correct:.2f}")

# Дополнительно: recall по параграфам (сколько процентов от всех правильных параграфов найдено)
recall_retriever = total_retrieved_correct_sum / total_correct_count_sum * 100
recall_model = total_found_correct_sum / total_correct_count_sum * 100
print(f"Recall ретривера: {recall_retriever:.2f}%")
print(f"Recall модели: {recall_model:.2f}%")

=== МЕТРИКИ ===
Всего вопросов: 100
Ретривер (top-30) нашёл все правильные параграфы в 36 вопросах (36.00%)
Модель (LLM) нашла все правильные параграфы в 18 вопросах (18.00%)
Среднее количество правильных параграфов на вопрос: 2.46
Среднее количество правильных параграфов, найденных ретривером: 1.51
Среднее количество правильных параграфов, найденных моделью: 0.73
Recall ретривера: 61.38%
Recall модели: 29.67%


## Error

In [392]:
import json

with open('results_qwen_4B_ENTROPY_THRESHOLD_2.json', 'r', encoding='utf-8') as f:
    results_by_main = json.load(f)

In [393]:
#results_by_main

In [394]:
import re
from collections import defaultdict

def analyze_subq_errors_by_hop(results_by_main):
    """
    Анализирует ошибки на подвопросах в зависимости от количества хопов.
    Возвращает словарь со статистикой.
    """
    hop_stats = defaultdict(lambda: {
        'main_total': 0,
        'main_correct': 0,
        'subq_total': 0,
        'subq_correct': 0,
        'initial_has_correct': {'total': 0, 'correct': 0},
        'initial_no_correct': {'total': 0, 'correct': 0},
        'error_examples': []   # первые 5 ошибок для демонстрации
    })
    
    for main_id, main_data in results_by_main.items():
        # Извлекаем количество хопов
        match = re.match(r'(\d+)hop', main_id)
        if not match:
            continue
        hop = int(match.group(1))
        
        if not isinstance(main_data, dict):
            continue
        
        subqs = main_data.get('subquestions', {})
        if not subqs:
            continue
        
        hop_stats[hop]['main_total'] += 1
        
        subq_correct_count = 0
        subq_total = 0
        
        for subq_id, subq_data in subqs.items():
            if not isinstance(subq_data, dict):
                continue
            subq_total += 1
            is_correct = subq_data.get('is_correct', False)
            
            if is_correct:
                subq_correct_count += 1
                hop_stats[hop]['subq_correct'] += 1
            else:
                # Сохраняем пример ошибки (не более 5 на хоп)
                if len(hop_stats[hop]['error_examples']) < 100:
                    hop_stats[hop]['error_examples'].append({
                        'main_id': main_id,
                        'subq_id': subq_id,
                        'found_doc': subq_data.get('found_doc'),
                        'correct_ids': subq_data.get('correct_ids', []),
                        'initial_has_correct': subq_data.get('initial_has_correct', False)
                    })
            
            hop_stats[hop]['subq_total'] += 1
            
            # Статистика по initial_has_correct
            init_has = subq_data.get('initial_has_correct', False)
            if init_has:
                hop_stats[hop]['initial_has_correct']['total'] += 1
                if is_correct:
                    hop_stats[hop]['initial_has_correct']['correct'] += 1
            else:
                hop_stats[hop]['initial_no_correct']['total'] += 1
                if is_correct:
                    hop_stats[hop]['initial_no_correct']['correct'] += 1
        
        # Проверяем, решён ли основной вопрос полностью верно
        if subq_correct_count == subq_total:
            hop_stats[hop]['main_correct'] += 1
    
    # Вывод статистики
    print("\n" + "="*70)
    print("СТАТИСТИКА ОШИБОК ПО КОЛИЧЕСТВУ ХОПОВ (на основе подвопросов)")
    print("="*70)
    
    for hop in sorted(hop_stats.keys()):
        s = hop_stats[hop]
        subq_acc = s['subq_correct'] / s['subq_total'] if s['subq_total'] else 0
        main_acc = s['main_correct'] / s['main_total'] if s['main_total'] else 0
        
        init_has_acc = (s['initial_has_correct']['correct'] / s['initial_has_correct']['total']
                        if s['initial_has_correct']['total'] else 0)
        init_no_acc = (s['initial_no_correct']['correct'] / s['initial_no_correct']['total']
                       if s['initial_no_correct']['total'] else 0)
        
        print(f"\n--- ХОП {hop} ---")
        print(f"  Основных вопросов: {s['main_total']}")
        print(f"  Подвопросов всего: {s['subq_total']}")
        print(f"  Точность на подвопросах: {subq_acc:.2%} ({s['subq_correct']}/{s['subq_total']})")
        print(f"  Основные вопросы решены полностью верно: {s['main_correct']} ({main_acc:.2%})")
        print(f"  Точность на подвопросах, где правильный документ был в начальной выдаче: {init_has_acc:.2%} "
              f"(из {s['initial_has_correct']['total']})")
        print(f"  Точность на подвопросах, где правильного документа не было в начальной выдаче: {init_no_acc:.2%} "
              f"(из {s['initial_no_correct']['total']})")
        
        if s['error_examples']:
            total_errors = s['subq_total'] - s['subq_correct']
            print(f"\n  Примеры ошибок (первые {len(s['error_examples'])} из {total_errors}):")
            for ex in s['error_examples']:
                print(f"    - Main: {ex['main_id']}, Subq: {ex['subq_id']}")
                print(f"      Найденный документ: {ex['found_doc']}, Правильные ID: {ex['correct_ids']}")
                print(f"      Правильный документ был в начальной выдаче: {ex['initial_has_correct']}")
            print()
    
    return hop_stats

# Пример использования:
# hop_stats = analyze_subq_errors_by_hop(results_by_main)

In [395]:
stats = analyze_subq_errors_by_hop(results_by_main)


СТАТИСТИКА ОШИБОК ПО КОЛИЧЕСТВУ ХОПОВ (на основе подвопросов)

--- ХОП 2 ---
  Основных вопросов: 63
  Подвопросов всего: 126
  Точность на подвопросах: 67.46% (85/126)
  Основные вопросы решены полностью верно: 31 (49.21%)
  Точность на подвопросах, где правильный документ был в начальной выдаче: 75.89% (из 112)
  Точность на подвопросах, где правильного документа не было в начальной выдаче: 0.00% (из 14)

  Примеры ошибок (первые 41 из 41):
    - Main: 2hop__13548_13529, Subq: To whom was Messi's goal in the first leg of the Copa del Rey compared?
      Найденный документ: No answer, Правильные ID: [122]
      Правильный документ был в начальной выдаче: True
    - Main: 2hop__787538_31270, Subq: What distance in miles is Clarksville , TN from Nashville?
      Найденный документ: No answer, Правильные ID: [238]
      Правильный документ был в начальной выдаче: True
    - Main: 2hop__256778_131879, Subq: Christopher Harris >> place of birth
      Найденный документ: No answer, Правиль

In [ ]:
3_hop + 1
2_hop + 2

In [529]:
import json

with open('results_qwen_9B_qwen.json', 'r', encoding='utf-8') as f:
    results_by_main = json.load(f)

In [530]:
import re
import json
from collections import defaultdict

def analyze_retrieval_errors_by_hop(results_by_main):
    """
    Анализирует ошибки алгоритма (неполное или неверное нахождение документов)
    в зависимости от количества хопов.
    """
    hop_stats = defaultdict(lambda: {
        'total': 0,
        'full_correct': 0,
        'partial': 0,
        'none_correct': 0,
        'error_examples': []   # первые 5 примеров неполных/неверных ответов
    })

    for main_id, data in results_by_main.items():
        # Извлекаем число хопов (2hop, 3hop1 и т.п. → 2 или 3)
        match = re.match(r'(\d+)hop', main_id)
        if not match:
            continue
        hop = int(match.group(1))

        if not isinstance(data, dict):
            continue

        total_correct = set(data.get('total_correct_ids', []))
        if not total_correct:
            continue  # нет правильных ID – некорректные данные

        # Получаем отправленные моделью ID из последнего submit_answer в step_info_history
        submitted_ids = set()
        step_history = data.get('step_info_history', [])
        for step in reversed(step_history):
            action = step.get('action', '')
            if action.startswith('submit_answer'):
                # Извлекаем список ID из строки, например 'submit_answer({"ids": [1, 11]})'
                try:
                    # Ищем часть после 'submit_answer('
                    args_str = action[len('submit_answer('):-1]  # убираем последнюю скобку
                    # Парсим как JSON (может быть {"id": 11} или {"ids": [1,11]})
                    args = json.loads(args_str)
                    if 'ids' in args:
                        submitted_ids = set(args['ids'])
                    elif 'id' in args:
                        submitted_ids = {args['id']}
                    else:
                        # Пустой словарь -> пустое множество
                        submitted_ids = set()
                    break
                except:
                    # Если не распарсилось, пробуем найти числа вручную
                    numbers = re.findall(r'\d+', args_str)
                    submitted_ids = set(map(int, numbers))
                    break

        # Определяем тип ответа
        found_correct = submitted_ids & total_correct
        if submitted_ids == total_correct:
            result_type = 'full_correct'
            hop_stats[hop]['full_correct'] += 1
        elif found_correct:
            result_type = 'partial'
            hop_stats[hop]['partial'] += 1
            # Сохраняем пример ошибки (не более 5)
            if len(hop_stats[hop]['error_examples']) < 5:
                hop_stats[hop]['error_examples'].append({
                    'main_id': main_id,
                    'question': data.get('main_question', '')[:120],
                    'submitted': sorted(submitted_ids),
                    'total_correct': sorted(total_correct),
                    'missing': sorted(total_correct - submitted_ids),
                    'extra': sorted(submitted_ids - total_correct)
                })
        else:
            result_type = 'none_correct'
            hop_stats[hop]['none_correct'] += 1
            if len(hop_stats[hop]['error_examples']) < 5:
                hop_stats[hop]['error_examples'].append({
                    'main_id': main_id,
                    'question': data.get('main_question', '')[:120],
                    'submitted': sorted(submitted_ids),
                    'total_correct': sorted(total_correct),
                    'missing': sorted(total_correct),
                    'extra': sorted(submitted_ids - total_correct)
                })

        hop_stats[hop]['total'] += 1

    # Вывод статистики
    print("\n" + "="*70)
    print("СТАТИСТИКА ОШИБОК ПО ХОПАМ (на основе найденных документов)")
    print("="*70)

    for hop in sorted(hop_stats.keys()):
        s = hop_stats[hop]
        full_acc = s['full_correct'] / s['total'] if s['total'] else 0
        partial_rate = s['partial'] / s['total'] if s['total'] else 0
        none_rate = s['none_correct'] / s['total'] if s['total'] else 0

        print(f"\n--- ХОП {hop} ---")
        print(f"  Всего вопросов: {s['total']}")
        print(f"  Полностью верные (все правильные ID найдены): {s['full_correct']} ({full_acc:.2%})")
        print(f"  Частично верные (найдена хотя бы часть): {s['partial']} ({partial_rate:.2%})")
        print(f"  Ни одного правильного ID не найдено: {s['none_correct']} ({none_rate:.2%})")

        if s['error_examples']:
            print(f"\n  Примеры ошибок (первые {len(s['error_examples'])}):")
            for ex in s['error_examples']:
                print(f"    - ID: {ex['main_id']}")
                print(f"      Вопрос: {ex['question']}")
                print(f"      Отправленные ID: {ex['submitted']}")
                print(f"      Правильные ID: {ex['total_correct']}")
                if ex['missing']:
                    print(f"      Не найдены (пропущены): {ex['missing']}")
                if ex['extra']:
                    print(f"      Лишние (неправильные): {ex['extra']}")
                print()
    return hop_stats

# Пример использования:
stats = analyze_retrieval_errors_by_hop(results_by_main)


СТАТИСТИКА ОШИБОК ПО ХОПАМ (на основе найденных документов)

--- ХОП 2 ---
  Всего вопросов: 63
  Полностью верные (все правильные ID найдены): 32 (50.79%)
  Частично верные (найдена хотя бы часть): 17 (26.98%)
  Ни одного правильного ID не найдено: 14 (22.22%)

  Примеры ошибок (первые 5):
    - ID: 2hop__423874_128023
      Вопрос: In which league was Ilsinho's team?
      Отправленные ID: [173]
      Правильные ID: [173, 178]
      Не найдены (пропущены): [178]

    - ID: 2hop__256778_131879
      Вопрос: Which is the body of water by the birthplace of Christopher Harris?
      Отправленные ID: []
      Правильные ID: [269, 274]
      Не найдены (пропущены): [269, 274]

    - ID: 2hop__826864_17335
      Вопрос: When did the owner of Bucephalus die?
      Отправленные ID: [314]
      Правильные ID: [308, 314]
      Не найдены (пропущены): [308]

    - ID: 2hop__223823_29905
      Вопрос: How many mandatory transmitters of the Canadian Broadcasting Centre's owner were updated before